<a href="https://colab.research.google.com/github/chenwh0/AudioSentimentAnalysisEnd2EndPipeline/blob/main/Stage2PipelineEvaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing Libraries

In [ ]:
!pip install -U transformers accelerate tqdm jiwer -q

In [ ]:
from datasets import load_dataset
import torch
from transformers import pipeline
import pandas
from tqdm.auto import tqdm
from sklearn.metrics import classification_report

import numpy
from huggingface_hub import HfApi
import time
import jiwer
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score, classification_report
import json

# Setup HERE

In [ ]:
folder_path = "AudioSentimentProject/" # CHANGE TO YOUR OWN PROJECT DIR

In [ ]:
device = 0 if torch.cuda.is_available() else -1

# **Stage 2**

## Data splits (Don't run again after 1st time!!!)

In [ ]:
# Create version
api = HfApi()
dataset_info = api.dataset_info("DynamicSuperb/Sentiment_Analysis_SLUE-VoxCeleb")
revision = dataset_info.sha
revision_filepath = f"{folder_path}dataset_revision.txt"
with open(revision_filepath, "w") as f:
    f.write(revision)
print("Dataset revision:", revision)

Dataset revision: 554ad4367e98b7c6f4d4d9756dc6bbdf345e042e


In [ ]:
# Load exact dataset revision
try:
    dataset = load_dataset("DynamicSuperb/Sentiment_Analysis_SLUE-VoxCeleb", split="test", revision=revision)
    original_dataframe = pandas.DataFrame(dataset)
except Exception as e:
    print(f"Failed to load dataset: {e}")
    exit()

In [ ]:
# Exclude samples labeled Disagreement & <mixed>
EXCLUDED_LABELS = ["Disagreement", "<mixed>"]
filtered_dataframe = original_dataframe[~original_dataframe["label"].isin(EXCLUDED_LABELS)].copy()
filtered_dataframe["original_index"] = filtered_dataframe.index

print("ORIGINAL dataset:")
print(original_dataframe["label"].value_counts())
print("\nFILTERED dataset:")
print(filtered_dataframe["label"].value_counts())

assert len(filtered_dataframe) == 3426
assert not filtered_dataframe["label"].isin(EXCLUDED_LABELS).any()

ORIGINAL dataset:
label
Neutral         2521
Positive         737
Negative         168
Disagreement     104
<mixed>           23
Name: count, dtype: int64

FILTERED dataset:
label
Neutral     2521
Positive     737
Negative     168
Name: count, dtype: int64


In [ ]:
random_generator = numpy.random.default_rng(8001)
seed1 = int(random_generator.integers(0, 2**32))
seed2 = int(random_generator.integers(0, 2**32))

# First split: 500 development, 2,926 remaining
development_slice, remaining = train_test_split(filtered_dataframe,
    train_size=500, stratify=filtered_dataframe["label"],
    random_state=seed1)

# Second split: 1,500 test, 1,426 load pool
test_slice, load_pool = train_test_split(remaining,
    train_size=1500,stratify=remaining["label"],
    random_state=seed2
)

# Get original dataset indicies
development_indices = development_slice["original_index"].to_numpy()
test_indices = test_slice["original_index"].to_numpy()
load_pool_indices = load_pool["original_index"].to_numpy()

development_slice_size = len(development_slice)
test_slice_size = len(test_slice)
load_pool_size = len(load_pool)
total_size = development_slice_size + test_slice_size + load_pool_size
# Verify sizes
print(f"""Development slice size = {development_slice_size}
Test slice size = {test_slice_size}
Load pool size = {load_pool_size}
Total size = {total_size}
""")
assert development_slice_size == 500
assert test_slice_size == 1500
assert load_pool_size == 1426
assert total_size == 3426
print("All slice size checks passed!")

# Verify no overlapping indices
assert set(development_indices).isdisjoint(test_indices)
assert set(development_indices).isdisjoint(load_pool_indices)
assert set(test_indices).isdisjoint(load_pool_indices)
print("All overlap checks passed!")

# Save all  indices
numpy.save(f"{folder_path}development_indices.npy", numpy.array(development_indices))
numpy.save(f"{folder_path}test_indices.npy", numpy.array(test_indices))
numpy.save(f"{folder_path}load_pool_indices.npy", numpy.array(load_pool_indices))

Development slice size = 500
Test slice size = 1500
Load pool size = 1426
Total size = 3426

All slice size checks passed!
All overlap checks passed!


## For future data retrieval (run again)

In [ ]:
# Read previously saved revision
with open(f"{folder_path}dataset_revision.txt", "r") as f:
    revision = f.read().strip()
print("Loading dataset revision:", revision)

# Load exact dataset version
dataset = load_dataset("DynamicSuperb/Sentiment_Analysis_SLUE-VoxCeleb", split="test", revision=revision)
original_dataframe = pandas.DataFrame(dataset)


Loading dataset revision: 554ad4367e98b7c6f4d4d9756dc6bbdf345e042e


README.md:   0%|          | 0.00/392 [00:00<?, ?B/s]

data/test-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  270MB            

data/test-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/test-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  251MB            

data/test-00001-of-00002.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/3553 [00:00<?, ? examples/s]

In [ ]:
# Exclude samples labeled Disagreement & <mixed>
EXCLUDED_LABELS = ["Disagreement", "<mixed>"]
filtered_dataframe = original_dataframe[~original_dataframe["label"].isin(EXCLUDED_LABELS)].copy()
filtered_dataframe["original_index"] = filtered_dataframe.index

print("ORIGINAL dataset:")
print(original_dataframe["label"].value_counts())
print("\nFILTERED dataset:")
print(filtered_dataframe["label"].value_counts())

assert len(filtered_dataframe) == 3426
assert not filtered_dataframe["label"].isin(EXCLUDED_LABELS).any()

ORIGINAL dataset:
label
Neutral         2521
Positive         737
Negative         168
Disagreement     104
<mixed>           23
Name: count, dtype: int64

FILTERED dataset:
label
Neutral     2521
Positive     737
Negative     168
Name: count, dtype: int64


In [ ]:
# Get original dataset indicies
development_indices = numpy.load(f"{folder_path}/development_indices.npy")
test_indices = numpy.load(f"{folder_path}/test_indices.npy")
load_pool_indices = numpy.load(f"{folder_path}/load_pool_indices.npy")

# Reconstruct data slices
development_slice = filtered_dataframe[filtered_dataframe["original_index"].isin(development_indices)].copy()
test_slice = filtered_dataframe[filtered_dataframe["original_index"].isin(test_indices)].copy()
load_pool = filtered_dataframe[filtered_dataframe["original_index"].isin(load_pool_indices)].copy()

development_slice_size = len(development_slice)
test_slice_size = len(test_slice)
load_pool_size = len(load_pool)
total_size = development_slice_size + test_slice_size + load_pool_size

# Verify sizes
print(f"""Development slice size = {development_slice_size}
Test slice size = {test_slice_size}
Load pool size = {load_pool_size}
Total size = {total_size}
""")
assert development_slice_size == 500
assert test_slice_size == 1500
assert load_pool_size == 1426
assert total_size == 3426
print("All slice size checks passed!")

# Verify no overlapping indices
assert set(development_indices).isdisjoint(test_indices)
assert set(development_indices).isdisjoint(load_pool_indices)
assert set(test_indices).isdisjoint(load_pool_indices)
print("All overlap checks passed!")

Development slice size = 500
Test slice size = 1500
Load pool size = 1426
Total size = 3426

All slice size checks passed!
All overlap checks passed!


## Baseline model (always predict "neutral")

In [ ]:
def baseline_sentiment_model(dataframe: pandas.DataFrame): # Always predict neutral
    return ["Neutral"] * len(dataframe)

test_slice["baseline_prediction"] = baseline_sentiment_model(test_slice)
print("\nBASELINE MODEL (always predict neutral)\n========================================")
print(classification_report(test_slice["label"], test_slice["baseline_prediction"], zero_division=0))
print()


BASELINE MODEL (always predict neutral)
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00        74
     Neutral       0.74      1.00      0.85      1104
    Positive       0.00      0.00      0.00       322

    accuracy                           0.74      1500
   macro avg       0.25      0.33      0.28      1500
weighted avg       0.54      0.74      0.62      1500




## ASR quality

In [ ]:
def normalize_text(text):
    text = str(text).lower().translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()

def calculate_wer(df, ref_col, hyp_col):
    ref = df[ref_col].fillna("").map(normalize_text)
    hyp = df[hyp_col].fillna("").map(normalize_text)
    keep = ref != ""                                  # jiwer can't score empty references
    corpus_wer = jiwer.wer(ref[keep].tolist(), hyp[keep].tolist())
    per_clip = pandas.Series(
        [jiwer.wer(r, h) for r, h in zip(ref[keep], hyp[keep])], index=ref[keep].index)
    print(f"{hyp_col}: corpus WER = {corpus_wer:.4f} "
          f"({(~keep).sum()} empty references skipped)")
    return corpus_wer, per_clip

## Gold input ablation
Pseudo-reference/gold transcript:
Use Whisper-large-v3 to transcribe the "true" transcripts.

In [ ]:
def get_asr_model(model_name):
    kwargs = {"chunk_length_s": 30} if "whisper" in model_name else {}
    return pipeline("automatic-speech-recognition", model=model_name, device=device, **kwargs)

def transcribe_audio(asr_model, audio):
    start = time.perf_counter()

    samples = audio.get_all_samples()
    waveform = samples.data.squeeze().numpy()
    sample_rate = samples.sample_rate
    result = asr_model({"raw": waveform, "sampling_rate": sample_rate})
    transcript = result["text"]

    end = time.perf_counter()
    transcribe_time_ms = (end - start) * 1000
    return transcript, transcribe_time_ms

REF = "openai/whisper-large-v3"
ASR_MODELS = [REF, # for Pseudo-reference transcripts
                "AventIQ-AI/whisper-audio-to-text",
                "facebook/s2t-small-librispeech-asr"]
wer_rows = []

for asr_model_name in ASR_MODELS:
    asr_model = get_asr_model(asr_model_name)

    development_results = development_slice["audio"].apply(lambda audio: transcribe_audio(asr_model, audio))
    development_slice[f"{asr_model_name}_transcript"] = development_results.apply(lambda x: x[0])
    development_slice[f"{asr_model_name}_transcribe_time_ms"] = development_results.apply(lambda x: x[1])

    test_results = test_slice["audio"].apply(lambda audio: transcribe_audio(asr_model, audio))
    test_slice[f"{asr_model_name}_transcript"] = test_results.apply(lambda x: x[0])
    test_slice[f"{asr_model_name}_transcribe_time_ms"] = test_results.apply(lambda x: x[1])

    if asr_model_name != REF:
        ref_col = f"{REF}_transcript"

        corpus_wer, per_clip = calculate_wer(test_slice, ref_col, f"{asr_model_name}_transcript")
        test_slice[f"{asr_model_name}_wer"] = per_clip            # one column per model, no overwriting
        wer_rows.append({"asr": asr_model_name, "wer": corpus_wer})
    del asr_model
    torch.cuda.empty_cache()

pandas.DataFrame(wer_rows).to_csv(f"{folder_path}wer_test.csv", index=False)
exclude_columns = ["audio"]
for df in (development_slice, test_slice):
    df["duration"] = df["audio"].apply(lambda a: a.metadata.duration_seconds)
development_slice.drop(columns=exclude_columns).to_csv(f"{folder_path}development.csv", index=False)
test_slice.drop(columns=exclude_columns).to_csv(f"{folder_path}test.csv", index=False)

In [ ]:
def tune_confidence_threshold(dataframe, prediction_column, confidence_column, label_column="label"):

    # Actual correctness of sentiment predictions
    correctness = (dataframe[prediction_column] == dataframe[label_column]).astype(int)

    threshold_results = {}

    for threshold in numpy.arange(0.05, 1.00, 0.05):
        threshold = round(threshold, 2)

        confidence_check = (dataframe[confidence_column] >= threshold).astype(int)
        threshold_results[threshold] = balanced_accuracy_score(correctness, confidence_check)

    best_threshold = max(threshold_results, key=threshold_results.get)
    best_balanced_accuracy = threshold_results[best_threshold]
    return best_threshold, best_balanced_accuracy, threshold_results

def get_sentiment_analysis_model(model_name):
    return pipeline("sentiment-analysis", model=model_name, device=device)

def predict_sentiment(sentiment_model, transcription: str):
    start = time.perf_counter()
    result = sentiment_model(transcription)[0]
    end = time.perf_counter()
    sentiment_time_ms = (end - start) * 1000
    return result["label"], result["score"], sentiment_time_ms

# Relabel mapping
def map_siebert_label(label, score, threshold):
    if score < threshold:
        return "Neutral"
    return label.capitalize()

multilingual_label_mapping = {
    "very negative": "Negative",
    "negative": "Negative",
    "neutral": "Neutral",
    "positive": "Positive",
    "very positive": "Positive"
}

# ASSERTIONS
def assert_clips(dataframe: pandas.DataFrame, transcript_column: str="asr_transcription", duration_column: str="duration", confidence_column: str="score", threshold: float=0.7):
    def is_non_empty_transcript(transcript):
        return len(str(transcript).split()) >= 2
    def is_plausible_speech_rate(transcript, duration):
        word_count = len(str(transcript).split())
        if duration <= 0:
            return False
        speech_rate = word_count / duration
        return 0.5 <= speech_rate <= 5.0
    def is_confident_classification(confidence):
        return confidence >= threshold
    dataframe["assert_non_empty_transcript"] = dataframe[transcript_column].apply(is_non_empty_transcript)
    dataframe["assert_plausible_speech_rate"] = dataframe.apply(lambda row:
                                                                is_plausible_speech_rate(row[transcript_column], row[duration_column]),
                                                                axis=1)
    dataframe["assert_confident_classification"] = dataframe[confidence_column].apply(is_confident_classification)
    dataframe["assert_passed"] = dataframe[["assert_non_empty_transcript", "assert_plausible_speech_rate", "assert_confident_classification"]].all(axis=1)
    return dataframe

def map_labels(model, raw, score, tau=None):
    if "cardiff" in model:      return raw.str.capitalize()
    if "multilingual" in model: return raw.str.lower().map(multilingual_label_mapping)
    return pandas.Series([map_siebert_label(l, s, tau) for l, s in zip(raw, score)], index=raw.index)

def run_clf(clf, texts):
    idx = [i for i, t in zip(texts.index, texts) if str(t).strip()]
    outs = clf([texts[i] for i in idx], truncation=True, max_length=512, batch_size=32)
    lab = pandas.Series("neutral", index=texts.index); sc = pandas.Series(0.0, index=texts.index)
    for i, o in zip(idx, outs): lab[i], sc[i] = o["label"], o["score"]
    return lab, sc

SENTIMENT_MODELS = [
    "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "tabularisai/multilingual-sentiment-analysis",
    "siebert/sentiment-roberta-large-english"
]

REF = "openai/whisper-large-v3"
results, thresholds, gold_cache = [], {}, {}
assert set(test_slice["label"]) == {"Negative", "Neutral", "Positive"}, test_slice["label"].unique()

for s_name in SENTIMENT_MODELS:
    clf = get_sentiment_analysis_model(s_name)
    for a_name in [m for m in ASR_MODELS if m != REF]:
        cfg = f"{a_name}|{s_name}"
        for src, tcol in [("full", f"{a_name}_transcript"), ("gold", f"{REF}_transcript")]:
            if src == "gold" and s_name in gold_cache:   # 2) first lines inside the inner loop
                results.append({**gold_cache[s_name], "config": cfg})
                continue
            dev_raw, dev_sc = run_clf(clf, development_slice[tcol])
            tst_raw, tst_sc = run_clf(clf, test_slice[tcol])
            tau = None
            if "siebert" in s_name:   # tune tau on dev macro-F1
                grid = [round(x, 2) for x in numpy.arange(0.50, 1.00, 0.01)]
                tau = max(grid, key=lambda g: f1_score(development_slice["label"],
                          map_labels(s_name, dev_raw, dev_sc, g), average="macro"))
            dev_pred = map_labels(s_name, dev_raw, dev_sc, tau)
            tst_pred = map_labels(s_name, tst_raw, tst_sc, tau)
            y, yd = test_slice["label"], development_slice["label"]

            row = dict(config=cfg, input=src, tau=tau,
                       macro_f1=f1_score(y, tst_pred, average="macro"),
                       accuracy=accuracy_score(y, tst_pred))
            if src == "full":
                # tune t on dev (common rule)
                correct = (dev_pred == yd).astype(int)
                grid = [round(x, 2) for x in numpy.arange(0.05, 1.00, 0.05)]
                t = max(grid, key=lambda g: balanced_accuracy_score(correct, (dev_sc >= g).astype(int)))
                thresholds[cfg] = t
                tr = test_slice[tcol].fillna("")
                nwords = tr.str.split().str.len()
                a1 = nwords >= 2
                a2 = (nwords / test_slice["duration"]).between(0.5, 5.0)
                a3 = tst_sc >= t
                row.update(t=t, A1=a1.mean(), A2=a2.mean(), A3=a3.mean(),
                           assert_pass=(a1 & a2 & a3).mean())
                rep = classification_report(y, tst_pred, zero_division=0, output_dict=True)
                name = f"{a_name.split('/')[-1]}__{s_name.split('/')[-1]}"
                pandas.DataFrame(rep).T.to_csv(f"{folder_path}report_{name}.csv")

            results.append(row)
            if src == "gold":
                gold_cache[s_name] = row

res = pandas.DataFrame(results)
res.to_csv(f"{folder_path}m2_t1_results.csv", index=False)

# Agreement between assertion pass rate and true accuracy
full = res[res.input == "full"]
print("Pearson:", full["assert_pass"].corr(full["macro_f1"]),
      "Spearman:", full["assert_pass"].corr(full["macro_f1"], method="spearman"))
gold = res[res.input == "gold"].set_index("config")["macro_f1"]
full = full.set_index("config"); full["upstream_loss"] = gold - full["macro_f1"]
full[["macro_f1", "accuracy", "assert_pass", "upstream_loss"]].to_csv(f"{folder_path}m2_t1_summary.csv")
print(full[["macro_f1", "assert_pass", "upstream_loss"]])

## Request & SLA unit

In [ ]:
def calculate_latency_metrics(dataframe, processing_column):

    processing_time = dataframe[processing_column]

    duration = dataframe["duration"]

    rtf = (processing_time / 1000) / duration

    return {
        "mean_latency_ms": processing_time.mean(),
        "median_latency_ms": processing_time.median(),
        "p95_latency_ms": processing_time.quantile(0.95),
        "mean_rtf": rtf.mean(),
        "median_rtf": rtf.median(),
        "p95_rtf": rtf.quantile(0.95)
    }

In [ ]:
calculate_latency_metricst()